In [ ]:
#Connecting to Remote Cloud Data

# Spark JARs: We pull in Sedona and Hadoop AWS packages so we can handle spatial ops and remote files.
# S3 endpoint: We’re connecting to a custom HTTPS-based S3-compatible endpoint, not AWS. Sedona doesn’t 
# care—it just needs to know how to talk to it.
# Memory tuning: Even locally, bumping driver/executor memory can help when working with larger datasets.

In [3]:
from sedona.spark import SedonaContext

config = (
    SedonaContext.builder()
    
    # Connect to the JAR (Java ARchive) packages
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.sedona:sedona-spark-3.5_2.12:1.6.1",
            "org.datasyslab:geotools-wrapper:1.7.0-28.5",
            "org.apache.hadoop:hadoop-aws:3.3.2"
        ])
    )
    .config("spark.jars.repositories", "https://artifacts.unidata.ucar.edu/repository/unidata-all")
    
    # Connect to remote data on Source Cooperative - you will need to sign up for an account and get your access and secret keys
    
    .config("spark.hadoop.fs.s3a.endpoint", "https://data.source.coop") 
    .config("spark.hadoop.fs.s3a.access.key", "SOURCE_COOP_S3_ACCESS_KEY")
    .config("spark.hadoop.fs.s3a.secret.key", "SOURCE_COOP_S3_SECRET_KEY")
    
    # Enable S3 access
    
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    
    # You can add this if you want to use public S3 data, for know we use local  
    
    # .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
    #         "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider") 
    
    .config("spark.executor.memory", "12G")
    .config("spark.driver.memory", "12G")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

sedona = SedonaContext.create(config)
sedona.sparkContext.setLogLevel("ERROR")

# Allowing Spark to fetch resources (jars, data) directly from HTTPS endpoints 
sedona.conf.set("fs.https.impl", "org.apache.hadoop.fs.http.HttpsFileSystem")

https://artifacts.unidata.ucar.edu/repository/unidata-all added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fe498710-e2ac-416c-a2a5-627ce8e78c89;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.6.1 in central
	found org.apache.sedona#sedona-common;1.6.1 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.19.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2

In [5]:
# Creating simple object row inside a list

In [6]:
from pyspark.sql import Row

In [7]:
data = [
    Row(id = 1, name = "Point A", lat = 40.7128, lon = -74.0060),
    Row(id = 2, name = "Point B", lat = 34.0522, lon = -118.2437),
    Row(id = 3, name = "Point C", lat = 37.7749, lon = -122.4194)
]  

In [8]:
df = sedona.createDataFrame(data)
df.show()

+---+-------+-------+---------+
| id|   name|    lat|      lon|
+---+-------+-------+---------+
|  1|Point A|40.7128|  -74.006|
|  2|Point B|34.0522|-118.2437|
|  3|Point C|37.7749|-122.4194|
+---+-------+-------+---------+



In [9]:
from pyspark.sql.functions import expr

In [13]:
# Adding geom column
df_geom = df.withColumn("geom"
                        , expr("ST_Point(cast(lon as Decimal(24, 20)), cast(lat as Decimal(24, 20)))")
                       )

In [15]:
df_geom.show(truncate = False)

+---+-------+-------+---------+-------------------------+
|id |name   |lat    |lon      |geom                     |
+---+-------+-------+---------+-------------------------+
|1  |Point A|40.7128|-74.006  |POINT (-74.006 40.7128)  |
|2  |Point B|34.0522|-118.2437|POINT (-118.2437 34.0522)|
|3  |Point C|37.7749|-122.4194|POINT (-122.4194 37.7749)|
+---+-------+-------+---------+-------------------------+



In [16]:
sql = """
SELECT ST_AreaSpheroid(
    ST_GeomFromWKT('Polygon ((34 35, 28 30, 25 34, 34 35))')
) as result
"""

sedona.sql(sql).show(truncate = False)

+---------------------+
|result               |
+---------------------+
|2.0182485081176245E11|
+---------------------+

